In [1]:
%cd ..
from pathlib import Path
import csv
from collections import Counter
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from Recognition.live_recognition import run_live_recognition

c:\Users\mjane\Documents\GitHub\LSTM-FAISS-DTW


c:\Users\mjane\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
results = run_live_recognition(
    is_diagnostic= True, 
    is_custom_database= True,
    is_sliding_window= False,
    labels= ["WORK", "SLEEP", "ME", "HOME", "READ", "YOU"],               #labels=None (all), labels=["BOOK", "READ"] (selected)
    repeats= 3,
    shuffle= True,
    camera_index= 0,
    top_k= 1,
    model = "Transformer"
)

Model works on: cpu
FAISS index: 10 vectors, 10 gesture classes
[1/18] true=READ pred=READ correct=True score=0.06 margin=0.10
[2/18] true=READ pred=READ correct=True score=0.08 margin=0.13
[3/18] true=ME pred=ME correct=True score=0.89 margin=0.71
[4/18] true=SLEEP pred=SLEEP correct=True score=0.43 margin=0.01
[5/18] true=ME pred=ME correct=True score=0.85 margin=0.71
[6/18] true=WORK pred=WORK correct=True score=0.90 margin=0.74
[7/18] true=YOU pred=YOU correct=True score=0.91 margin=0.74
[8/18] true=READ pred=READ correct=True score=0.08 margin=0.05
[9/18] true=HOME pred=HOME correct=True score=0.41 margin=0.04
[10/18] true=SLEEP pred=SLEEP correct=True score=0.45 margin=0.01
[11/18] true=HOME pred=YOU correct=False score=0.65 margin=0.46
[12/18] true=YOU pred=YOU correct=True score=0.91 margin=0.74
[13/18] true=SLEEP pred=SLEEP correct=True score=0.44 margin=0.01
[14/18] true=HOME pred=HOME correct=True score=0.32 margin=0.03
[15/18] true=WORK pred=YOU correct=False score=0.76 mar

In [6]:
data = pd.DataFrame(results)
y_true = data["true_label"]
y_pred = data["predicted_label"]

labels_names = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels_names)
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

fig = go.Figure(data=go.Heatmap(
    z = cm_norm,
    zmin = 0,
    zmax = 1,
    x = labels_names,
    y = labels_names,
    colorscale="turbo",
    text=cm,
    texttemplate="%{text}",
    hovertemplate="True: %{y}<br>Pred: %{x}<extra></extra>"
))

fig.update_layout(
    title="Prediction in Best Epoch",
    xaxis_title="Predicted label",
    yaxis_title="True label",
    width=800,
    height=800,
    margin=dict(l=150, r=50, t=80, b=150)
)

fig.show()

In [11]:
df_avg = data.groupby("true_label")[["score", "margin"]].mean().reset_index()

fig = go.Figure(data=[
    go.Bar(
        name='Avg Score', 
        x=df_avg['true_label'], 
        y=df_avg['score'],
        marker_color='green'
    ),
    go.Bar(
        name='Avg Margin', 
        x=df_avg['true_label'], 
        y=df_avg['margin'],
        marker_color='blue'
    )
])

fig.update_layout(
    title='Average Score and Margin per Class',
    xaxis_title='Classes',
    yaxis_title='AVG',
    barmode='group',
    template='plotly_white',
    legend_title='Metrics',
    hovermode='x unified'
)

fig.show()